# Weekend vs. Weekday Shopping: From a Sample to the Whole Population

**The question:** is there a significant difference between weekend and weekday shopper spending,
both in the average amount spent and in how much that spending varies?

**The data:** the UCI "Online Retail" dataset (Chen, D., 2015, CC BY 4.0), real, timestamped
transactions from a UK-based online retailer, Dec 2010-Dec 2011.

**What this project is about:** every study only ever sees a sample, never the whole population.
This notebook is fundamentally about how we extract information from a sample, and to what extent
that information can be trusted for the whole population. This notebook uses a real dataset large enough that we can check a
sample's answer against the actual population's answer directly. Along the way, we are also interested in a second,
related question: not just whether the two groups' *means* differ, but whether their *variances*
differ too, since that assumption drives which test is actually valid to use.

**The plan:**
1. In practice, when we collect data, we collect a sample of weekend and weekday buyers, and analyze
   that sample. We assume homogeneity of variance and use the pooled $t$-test.
2. We restrict ourselves to typical buyers, then re-sample. The raw sample data mixes ordinary
   shoppers with occasional large orders that distort the comparison.
3. Since the entire population is available here, compute its true mean and variance directly,
   and test $H_0$ against the population itself.
4. Alongside the pooled test, compute **Welch's test**, which does not assume variance homogeneity,
   such that the two tests can be compared directly throughout this notebook.

In [25]:
import numpy as np
import pandas as pd
from scipy import stats

## 0. Understand the data


In [42]:
url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
raw = pd.read_csv(url, encoding="latin1", dtype={"CustomerID": str})
raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.5500,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.3900,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.7500,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.3900,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.3900,17850,United Kingdom


**How big is the data?** 

In [43]:
raw.shape

(541909, 8)

**What is the data's type?**

In [44]:
raw.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID      object
Country         object
dtype: object

### Label each row Weekend or Weekday

`InvoiceDate` is currently `object` (plain text), not a real date. It needs to be converted to a datetime first.
Once converted, `.dt.day_name()` pulls out each row's actual day (e.g. "Monday", "Sunday"), which then gets
collapsed into the two categories: Weekend and Weekday.

In [45]:
# Convert InvoiceDate from plain text into a real datetime, so date-related operations become possible
raw["InvoiceDate"] = pd.to_datetime(raw["InvoiceDate"], format="%m/%d/%Y %H:%M")

# .dt.day_name() is the one place a date-specific tool is needed: it reads the real datetime
# and returns the actual day of the week as plain text (e.g. "Monday", "Sunday")
raw["DayOfWeek"] = raw["InvoiceDate"].dt.day_name()

# From here on, DayOfWeek is just ordinary text — no more date tools needed,
# just a plain string comparison to collapse 7 day names into 2 categories
day_types = []
for day in raw["DayOfWeek"]:
    if day == "Saturday" or day == "Sunday":
        day_types.append("Weekend")
    else:
        day_types.append("Weekday")
raw["DayType"] = day_types

raw["DayType"].value_counts()

DayType
Weekday    477534
Weekend     64375
Name: count, dtype: int64

#### Check the type of invoices

Check what prefixes appear in the data.

In [46]:
invoice = raw["InvoiceNo"].astype(str)
first_char = invoice.str[0]
first_char.value_counts()


InvoiceNo
5    532618
C      9288
A         3
Name: count, dtype: int64

Three types of invoice numbers exist: ordinary
numeric invoices (532,618 of them), invoices starting with **C** (9,288), and a small number
starting with **A** (just 3). 

In [47]:
raw[raw["InvoiceNo"].astype(str).str.startswith("A")]


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,DayOfWeek,DayType
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,"11,062.0600",NaN,United Kingdom,Friday,Weekday
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,"-11,062.0600",NaN,United Kingdom,Friday,Weekday
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,"-11,062.0600",NaN,United Kingdom,Friday,Weekday


These are internal **accounting adjustments** ("Adjust bad debt"), not real purchases. Nothing about these rows reflects shopping behavior. Therefore, these will not be counted as orders.

In [49]:
raw[raw["InvoiceNo"].astype(str).str.startswith("C")].head(5)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,DayOfWeek,DayType
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.5000,14527,United Kingdom,Wednesday,Weekday
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.6500,15311,United Kingdom,Wednesday,Weekday
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.6500,17548,United Kingdom,Wednesday,Weekday
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.2900,17548,United Kingdom,Wednesday,Weekday
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.2900,17548,United Kingdom,Wednesday,Weekday


These are **cancellations**: each of them has a negative `Quantity`. Again, these do not reflect purchases.

In [50]:
ordinary_purchase = ~invoice.str.startswith(("C", "A"))
clean = raw[ordinary_purchase]
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)]
clean.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,DayOfWeek,DayType
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.5500,17850,United Kingdom,Wednesday,Weekday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom,Wednesday,Weekday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.7500,17850,United Kingdom,Wednesday,Weekday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom,Wednesday,Weekday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.3900,17850,United Kingdom,Wednesday,Weekday
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.6500,17850,United Kingdom,Wednesday,Weekday
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.2500,17850,United Kingdom,Wednesday,Weekday
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.8500,17850,United Kingdom,Wednesday,Weekday
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.8500,17850,United Kingdom,Wednesday,Weekday
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.6900,13047,United Kingdom,Wednesday,Weekday


#### Collapse line items into one row per order

We note that the `clean` data set has one row per **product**, not one row per **order** — a single order can span
several rows, all sharing the same `InvoiceNo` (see the first 7 rows above, all `InvoiceNo=536365`).
We only care about **orders**, not individual products, so these product-level rows
need to be grouped and summed by `InvoiceNo` into one row per order, each with a single total dollar value.

In [51]:
clean["LineTotal"] = clean["Quantity"] * clean["UnitPrice"]

orders = clean.groupby("InvoiceNo").agg(
    OrderValue=("LineTotal", "sum"),
    InvoiceDate=("InvoiceDate", "first"),
    DayType=("DayType", "first"),
).reset_index()

orders.head(10)


,InvoiceNo,OrderValue,InvoiceDate,DayType
0,536365,139.1200,2010-12-01 08:26:00,Weekday
1,536366,22.2000,2010-12-01 08:28:00,Weekday
2,536367,278.7300,2010-12-01 08:34:00,Weekday
3,536368,70.0500,2010-12-01 08:34:00,Weekday
4,536369,17.8500,2010-12-01 08:35:00,Weekday
5,536370,855.8600,2010-12-01 08:45:00,Weekday
6,536371,204.0000,2010-12-01 09:00:00,Weekday
7,536372,22.2000,2010-12-01 09:01:00,Weekday
8,536373,259.8600,2010-12-01 09:02:00,Weekday
9,536374,350.4000,2010-12-01 09:09:00,Weekday


## Step 1 — Raw Sample analysis

This is what an ordinary analysis does first: draw a random sample and analyze it. We draw $n_1=n_2=500$ orders.

In [52]:
weekend_orders = orders.loc[orders["DayType"] == "Weekend", "OrderValue"]
weekday_orders = orders.loc[orders["DayType"] == "Weekday", "OrderValue"]

rng = np.random.default_rng(2)
weekend = rng.choice(weekend_orders.values, size=500, replace=False)
weekday = rng.choice(weekday_orders.values, size=500, replace=False)

n1, n2 = len(weekend), len(weekday)
x1, x2 = weekend.mean(), weekday.mean()
s1, s2 = weekend.std(ddof=1), weekday.std(ddof=1)

print(f"Weekend: n={n1}, mean=${x1:.2f}, sd=${s1:.2f}")
print(f"Weekday: n={n2}, mean=${x2:.2f}, sd=${s2:.2f}")

Weekend: n=500, mean=$368.86, sd=$382.24
Weekday: n=500, mean=$458.64, sd=$686.09


#### What `s1`/`s2` are actually computing

$$S = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar x)^2}$$

For each order value $x_i$ in a group, $(x_i-\bar x)$ is how far that order sits from its group's
own mean. Summing across all $n$ orders and dividing by $n-1$ gives the
average squared deviation, an unbiased estimate of the population variance; the $n-1$ correction
(rather than $n$) accounts for the fact that one degree of freedom is already used up computing
$\bar x$ from the same data. 

This is exactly what `.std(ddof=1)` computes: `ddof=1` is what makes us divide by
$n-1$ instead of the default $n$.

#### $H_0: \mu_{\text{weekend}} = \mu_{\text{weekday}} \quad H_a: \mu_{\text{weekend}} \neq \mu_{\text{weekday}}$

In [57]:
# Pooled test (assumes equal variances)
sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)   # pooled variance
sp = np.sqrt(sp2)
se = sp * np.sqrt(1/n1 + 1/n2)
t_stat = (x1 - x2) / se
dof = n1 + n2 - 2
p_value = 2 * stats.t.sf(abs(t_stat), dof)

print(f"Pooled variance (Sp^2): {sp2:.4f}")
print(f"t-statistic:            {t_stat:.4f}")
print(f"Degrees of freedom:     {dof}")
print(f"Attained p-value:       {p_value:.4g}")

t_check, p_check = stats.ttest_ind(weekend, weekday, equal_var=True)
print(f"scipy check: t={t_check:.4f}, p={p_check:.4f}  (matches manual calc: {np.isclose(t_stat, t_check)})")

# Welch's test (does not assume equal variances)
s1, s2 = weekend.std(ddof=1), weekday.std(ddof=1)
se_welch = np.sqrt(s1**2/n1 + s2**2/n2)
t_welch = (x1 - x2) / se_welch
v = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
p_welch = 2 * stats.t.sf(abs(t_welch), v)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch:.4f}")
print(f"Degrees of freedom:     {v:.2f}")
print(f"Attained p-value:       {p_welch:.4g}")

t_welch_check, p_welch_check = stats.ttest_ind(weekend, weekday, equal_var=False)
print(f"scipy check: t={t_welch_check:.4f}, p={p_welch_check:.4f}  (matches manual calc: {np.isclose(t_welch, t_welch_check)})")

Pooled variance (Sp^2): 308410.6311
t-statistic:            -2.5561
Degrees of freedom:     998
Attained p-value:       0.01073
scipy check: t=-2.5561, p=0.0107  (matches manual calc: True)

Welch's t-test:
t-statistic:            -2.5561
Degrees of freedom:     781.55
Attained p-value:       0.01077
scipy check: t=-2.5561, p=0.0108  (matches manual calc: True)


### Variance problem

Weekday $sd=\$686.09$ is close to the weekday mean ($\$458.64$) — for a variable bounded below by
zero, such as spending, $sd$ approaching or exceeding the mean is inconsistent with a roughly-normal,
well-behaved distribution.

The cause is that this dataset combines ordinary shoppers with occasional **large orders**. A small
number of such orders inflate both the mean and the variance substantially, and the $p$-value
obtained above (0.01073) is affected by them. Because the variance is this large, we would like to
restrict the distribution to typical buyers.

## Step 2 — Restrict to typical shoppers

To answer the original question honestly, extreme orders are trimmed using the standard IQR rule
($Q_3+1.5\times\text{IQR}$): the same shared cutoff applied to both groups, computed from the
combined sample.

In [35]:
sample_values = np.concatenate([weekend, weekday])   # the n1+n2 = 1000 real orders from the sample
q1, q3 = np.percentile(sample_values, [25, 75])
cutoff = q3 + 1.5 * (q3 - q1)

# apply the SAME shared cutoff to each group separately, so groups stay identifiable
weekend_trimmed = weekend[weekend <= cutoff]
weekday_trimmed = weekday[weekday <= cutoff]
sample_trimmed = np.concatenate([weekend_trimmed, weekday_trimmed])

q1_trimmed, q3_trimmed = np.percentile(sample_trimmed, [25, 75])

print(f"Mean before trimming: ${sample_values.mean():.2f}")
print(f"Mean after trimming:  ${sample_trimmed.mean():.2f}")
print(f"Q1 (trimmed):   ${q1_trimmed:.2f}")
print(f"Q3 (trimmed):   ${q3_trimmed:.2f}")
print(f"Cutoff: {cutoff:.2f}")
print(f"\nRemoved from weekend: {(weekend>cutoff).sum()}/{len(weekend)}")
print(f"Removed from weekday: {(weekday>cutoff).sum()}/{len(weekday)}")


Mean before trimming: $413.75
Mean after trimming:  $298.04
Q1 (trimmed):   $147.50
Q3 (trimmed):   $396.80
Cutoff: 921.86

Removed from weekend: 26/500
Removed from weekday: 56/500


#### Apply the cutoff and rerun the test


In [58]:
# use the same cutoff computed from the combined sample, but apply it to each group separately
weekend_trimmed = weekend[weekend <= cutoff]
weekday_trimmed = weekday[weekday <= cutoff]

n1, n2 = len(weekend_trimmed), len(weekday_trimmed)
x1, x2 = weekend_trimmed.mean(), weekday_trimmed.mean()
s1, s2 = weekend_trimmed.std(ddof=1), weekday_trimmed.std(ddof=1)

print(f"Weekend (trimmed): n={n1}, mean=${x1:.2f}, sd=${s1:.2f}")
print(f"Weekday (trimmed): n={n2}, mean=${x2:.2f}, sd=${s2:.2f}")

# H0: mu_weekend = mu_weekday   Ha: mu_weekend != mu_weekday

# Pooled test (assumes equal variances)
sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)
sp = np.sqrt(sp2)
se = sp * np.sqrt(1/n1 + 1/n2)
t_stat = (x1 - x2) / se
dof = n1 + n2 - 2
p_value = 2 * stats.t.sf(abs(t_stat), dof)

print(f"\nPooled variance (Sp^2): {sp2:.4f}")
print(f"t-statistic:            {t_stat:.4f}")
print(f"Degrees of freedom:     {dof}")
print(f"Attained p-value:       {p_value:.4g}")

t_check, p_check = stats.ttest_ind(weekend_trimmed, weekday_trimmed, equal_var=True)
print(f"scipy check: t={t_check:.4f}, p={p_check:.4f}  (matches manual calc: {np.isclose(t_stat, t_check)})")

# Welch's test (does not assume equal variances)
se_welch = np.sqrt(s1**2/n1 + s2**2/n2)
t_welch = (x1 - x2) / se_welch
v = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
p_welch = 2 * stats.t.sf(abs(t_welch), v)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch:.4f}")
print(f"Degrees of freedom:     {v:.2f}")
print(f"Attained p-value:       {p_welch:.4g}")

t_welch_check, p_welch_check = stats.ttest_ind(weekend_trimmed, weekday_trimmed, equal_var=False)
print(f"scipy check: t={t_welch_check:.4f}, p={p_welch_check:.4f}  (matches manual calc: {np.isclose(t_welch, t_welch_check)})")

Weekend (trimmed): n=477, mean=$308.80, sd=$204.56
Weekday (trimmed): n=456, mean=$308.57, sd=$227.72

Pooled variance (Sp^2): 46737.8199
t-statistic:            0.0159
Degrees of freedom:     931
Attained p-value:       0.9873
scipy check: t=0.0159, p=0.9873  (matches manual calc: True)

Welch's t-test:
t-statistic:            0.0159
Degrees of freedom:     910.08
Attained p-value:       0.9873
scipy check: t=0.0159, p=0.9873  (matches manual calc: True)


Much better-behaved: means (\$304.61 vs \$291.03) and SDs (\$198.28, \$203.79) are now on comparable
scales, and $mean-1\,sd$ is safely positive for both groups. The $t$-test's assumptions are far more
defensible here. On this sample, the difference is **not statistically significant** (pooled
$p\approx0.3067$, Welch $p\approx0.3072$ — the two nearly agree, since $n_1\approx n_2$ here).

## Step 3 — Population analysis

In a real experiment, the researcher gathers the data using the sample they can collect. The
sample size chosen above, 500/500, is a realistic, reasonably good sample. In our case, we happen to
have it all: every real order in the dataset, not just a sample of it. So instead of estimating the
population mean and variance, we can compute them directly, and run the same $H_0$ test on the full
population.

In [39]:
# --- Untrimmed (raw) full population, for comparison ---
weekend_pop_raw = orders.loc[orders.DayType == "Weekend", "OrderValue"].values
weekday_pop_raw = orders.loc[orders.DayType == "Weekday", "OrderValue"].values

n1_raw, n2_raw = len(weekend_pop_raw), len(weekday_pop_raw)
x1_raw, x2_raw = weekend_pop_raw.mean(), weekday_pop_raw.mean()
s1_raw, s2_raw = weekend_pop_raw.std(ddof=1), weekday_pop_raw.std(ddof=1)

print(f"Weekend population (raw): n={n1_raw:,}, mean=${x1_raw:.2f}, sd=${s1_raw:.2f}")
print(f"Weekday population (raw): n={n2_raw:,}, mean=${x2_raw:.2f}, sd=${s2_raw:.2f}")

# Pooled test (assumes equal variances)
sp2_raw = ((n1_raw - 1) * s1_raw**2 + (n2_raw - 1) * s2_raw**2) / (n1_raw + n2_raw - 2)
sp_raw = np.sqrt(sp2_raw)
se_raw = sp_raw * np.sqrt(1/n1_raw + 1/n2_raw)
t_stat_raw = (x1_raw - x2_raw) / se_raw
dof_raw = n1_raw + n2_raw - 2
p_value_raw = 2 * stats.t.sf(abs(t_stat_raw), dof_raw)

print(f"\nPooled variance (Sp^2): {sp2_raw:.4f}")
print(f"t-statistic:            {t_stat_raw:.4f}")
print(f"Degrees of freedom:     {dof_raw}")
print(f"Attained p-value:       {p_value_raw:.6f}  ({p_value_raw:.3g} to 3 s.f.)")

t_check_raw, p_check_raw = stats.ttest_ind(weekend_pop_raw, weekday_pop_raw, equal_var=True)
print(f"scipy check: t={t_check_raw:.4f}, p={p_check_raw:.4f}  (matches manual calc: {np.isclose(t_stat_raw, t_check_raw)})")

# Welch's test (does not assume equal variances)
se_welch_raw = np.sqrt(s1_raw**2/n1_raw + s2_raw**2/n2_raw)
t_welch_raw = (x1_raw - x2_raw) / se_welch_raw
v_raw = (s1_raw**2/n1_raw + s2_raw**2/n2_raw)**2 / ((s1_raw**2/n1_raw)**2/(n1_raw-1) + (s2_raw**2/n2_raw)**2/(n2_raw-1))
p_welch_raw = 2 * stats.t.sf(abs(t_welch_raw), v_raw)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch_raw:.4f}")
print(f"Degrees of freedom:     {v_raw:.2f}")
print(f"Attained p-value:       {p_welch_raw:.6f}  ({p_welch_raw:.3g} to 3 s.f.)")

t_welch_check_raw, p_welch_check_raw = stats.ttest_ind(weekend_pop_raw, weekday_pop_raw, equal_var=False)
print(f"scipy check: t={t_welch_check_raw:.4f}, p={p_welch_check_raw:.4f}  (matches manual calc: {np.isclose(t_welch_raw, t_welch_check_raw)})")

Weekend population (raw): n=2,204, mean=$369.25, sd=$540.61
Weekday population (raw): n=17,755, mean=$554.31, sd=$1875.52

Pooled variance (Sp^2): 3161535.6092
t-statistic:            -4.6085
Degrees of freedom:     19957
Attained p-value:       0.000004  (4.08e-06 to 3 s.f.)
scipy check: t=-4.6085, p=0.0000  (matches manual calc: True)

Welch's t-test:
t-statistic:            -10.1762
Degrees of freedom:     10731.02
Attained p-value:       0.000000  (3.26e-24 to 3 s.f.)
scipy check: t=-10.1762, p=0.0000  (matches manual calc: True)


### Restrict to typical buyers, at the population level

As before, it is worth restricting the population to typical buyers. Trimming the same way as Step 2, applied to the whole population, gives the a picture
of the typical buyer.

In [41]:
q1, q3 = orders["OrderValue"].quantile([0.25, 0.75])
cutoff = q3 + 1.5 * (q3 - q1)
print(f"Q1: ${q1:.2f}   Q3: ${q3:.2f}   Cutoff: ${cutoff:.2f}")

orders_trimmed = orders[orders["OrderValue"] <= cutoff].copy()

weekend_pop = orders_trimmed.loc[orders_trimmed.DayType == "Weekend", "OrderValue"].values
weekday_pop = orders_trimmed.loc[orders_trimmed.DayType == "Weekday", "OrderValue"].values

n1, n2 = len(weekend_pop), len(weekday_pop)
x1, x2 = weekend_pop.mean(), weekday_pop.mean()
s1, s2 = weekend_pop.std(ddof=1), weekday_pop.std(ddof=1)

print(f"Weekend population: n={n1:,}, mean=${x1:.2f}, sd=${s1:.2f}")
print(f"Weekday population: n={n2:,}, mean=${x2:.2f}, sd=${s2:.2f}")

# Pooled test (assumes equal variances)
sp2 = ((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)
sp = np.sqrt(sp2)
se = sp * np.sqrt(1/n1 + 1/n2)
t_stat = (x1 - x2) / se
dof = n1 + n2 - 2
p_value = 2 * stats.t.sf(abs(t_stat), dof)

print(f"\nPooled variance (Sp^2): {sp2:.4f}")
print(f"t-statistic:            {t_stat:.4f}")
print(f"Degrees of freedom:     {dof}")
print(f"Attained p-value:       {p_value:.6f}  ({p_value:.3g} to 3 s.f.)")

t_check, p_check = stats.ttest_ind(weekend_pop, weekday_pop, equal_var=True)
print(f"scipy check: t={t_check:.4f}, p={p_check:.4f}  (matches manual calc: {np.isclose(t_stat, t_check)})")

# Welch's test (does not assume equal variances)
se_welch = np.sqrt(s1**2/n1 + s2**2/n2)
t_welch = (x1 - x2) / se_welch
v = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1-1) + (s2**2/n2)**2/(n2-1))
p_welch = 2 * stats.t.sf(abs(t_welch), v)

print(f"\nWelch's t-test:")
print(f"t-statistic:            {t_welch:.4f}")
print(f"Degrees of freedom:     {v:.2f}")
print(f"Attained p-value:       {p_welch:.6f}  ({p_welch:.3g} to 3 s.f.)")

t_welch_check, p_welch_check = stats.ttest_ind(weekend_pop, weekday_pop, equal_var=False)
print(f"scipy check: t={t_welch_check:.4f}, p={p_welch_check:.4f}  (matches manual calc: {np.isclose(t_welch, t_welch_check)})")

Q1: $152.49   Q3: $495.50   Cutoff: $1010.01
Weekend population: n=2,117, mean=$298.24, sd=$202.98
Weekday population: n=16,032, mean=$308.31, sd=$219.54

Pooled variance (Sp^2): 47381.6737
t-statistic:            -2.0009
Degrees of freedom:     18147
Attained p-value:       0.045420  (0.0454 to 3 s.f.)
scipy check: t=-2.0009, p=0.0454  (matches manual calc: True)

Welch's t-test:
t-statistic:            -2.1248
Degrees of freedom:     2811.39
Attained p-value:       0.033690  (0.0337 to 3 s.f.)
scipy check: t=-2.1248, p=0.0337  (matches manual calc: True)


## Conclusions

The results below reflect the specific sample drawn in this run (seed=2); since sampling involves
randomness, re-running the code with a different seed, or drawing a fresh sample, would produce different 
numbers for the sample.

| Stage | n (per group) | mean gap | p (pooled) | p (Welch) 
|---|---|---|---|---|
| Raw sample | 500/500 | $\$89.78$ | 0.01073 | 0.01077 
| Trimmed sample | 474/444 | $\$13.58$ | 0.3067 | 0.3072 |
| Full population | 2,204/17,755 | $\$185.06$ | 4.08e-06 | 3.26e-24 | 
| Full trimmed population | 2,117/16,032 | $\$10.07$ | 0.0454 | 0.0337 |

**The original question:** is there sufficient evidence that weekend and weekday shoppers spend
differently, and what is the attained significance level?

**Observation 1:** a researcher working only with the trimmed sample (474/444) would compute
$p\approx0.31$ and could reasonably conclude "no evidence of a difference; weekend and weekday
shoppers are essentially the same." **This conclusion would be wrong**: at the full-population
scale, the exact same trimming procedure still yields $p<0.05$ (pooled $0.0454$, Welch $0.0337$), so
a real difference does exist. The correct conclusion from the sample alone should have been "not
enough evidence to detect a difference," not "no difference exists."

**Observation 2:** in the trimmed sample, weekend spending ($\$304.61$) came out *higher* than
weekday ($\$291.03$). In the trimmed population, it's the reverse; weekday ($\$308.31$) is higher
than weekend ($\$298.24$). A researcher trusting the sample wouldn't just under-detect a real effect;
they could report the wrong group as the bigger spender entirely.

**Observation 3 (how closely pooled and Welch agree tracks how balanced $n_1,n_2$ are, and that
divergence is itself informative):** on the raw sample, $n_1=n_2=500$ exactly, and the pooled and
Welch $t$-statistics come out numerically identical ($t=-2.5561$ both), only the degrees of freedom
differ (998 vs.\ 781.55). On the trimmed sample, $n_1=474$, $n_2=444$, close but not exactly equal,
and the $t$-statistics are close but no longer identical ($1.0227$ vs $1.0218$). At the population
level, $n_1\approx2{,}100$ against $n_2\approx18{,}000$, and the two tests diverge substantially.
Part of this is because $n_1$ and $n_2$ are very different: the pooled/Welch equivalence shown in the
attached PDF file holds when $n_1=n_2$. But the size of the divergence here is also a signal that the assumption underlying the pooled test
about **equal variance across weekend and weekday spending may not hold.**

---
#### Notes
* **Core Analysis:** All feature engineering, mathematical derivations, model training, and evaluation were developed by the author.
* **Editing Assistants:** Claude and Gemini were used to help refine text and code formatting.